# Live Data Fetch: Station Snapshot + Weather Forecast

Prototype for the daily ingestion step:
1. Fetch today's station snapshot from the CityBikes API
2. Inspect and validate the snapshot against the existing parquet schema
3. Fetch tomorrow's weather forecast from Open-Meteo
4. Validate the forecast matches the existing `weather_daily.parquet` schema
5. Save both outputs

In [8]:
import warnings
warnings.filterwarnings('ignore')

from datetime import date, timedelta
from pathlib import Path

import numpy as np
import openmeteo_requests
import pandas as pd
import requests
import requests_cache
from retry_requests import retry

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

ROOT         = Path('..').resolve()
SNAPSHOT_DIR = ROOT / 'bike_data_berlin'
WEATHER_PATH = ROOT / 'data' / 'processed' / 'weather_daily.parquet'

TIMEZONE      = 'Europe/Berlin'
CITYBIKES_URL = 'https://api.citybik.es/v2/networks/{}'
FORECAST_URL  = 'https://api.open-meteo.com/v1/forecast'
SYSTEMS       = {
    'nextbike-berlin' : 'nextbike-berlin',
    'callabike-berlin': 'callabike-berlin',
}
HOURLY_VARS = [
    'temperature_2m', 'precipitation', 'apparent_temperature',
    'rain', 'snowfall', 'weather_code', 'wind_speed_10m',
    'relative_humidity_2m', 'cloud_cover',
]

today    = date.today()
tomorrow = today + timedelta(days=1)
print(f'Today: {today}  |  Tomorrow: {tomorrow}')

Today: 2026-06-11  |  Tomorrow: 2026-06-12


## 1. Station Snapshot: CityBikes API

In [9]:
# Fetch raw JSON for one network and inspect the structure
network_id = 'nextbike-berlin'
resp = requests.get(CITYBIKES_URL.format(network_id), timeout=30)
resp.raise_for_status()
data = resp.json()

print('Top-level keys:', list(data.keys()))
print('Network keys  :', list(data['network'].keys()))
print('Station count :', len(data['network']['stations']))
print('\nFirst station:')
print(data['network']['stations'][0])

Top-level keys: ['network']
Network keys  : ['id', 'name', 'location', 'href', 'company', 'system', 'stations']
Station count : 1075

First station:
{'id': '00455239920d92cdc60461f8b8619fac', 'name': 'Linienstraße/Weydingerstraße', 'latitude': 52.527005, 'longitude': 13.414112, 'timestamp': '2026-06-11T09:12:20.591591+00:00Z', 'free_bikes': 3, 'empty_slots': 3, 'extra': {'uid': '138056052', 'number': '10740', 'slots': 6, 'bike_uids': ['18785', '16757', '19877'], 'virtual': True}}


In [10]:
# Parse all stations for both networks into the existing parquet schema:
#   tag, id, nuid, name, latitude, longitude, bikes, free, extra, timestamp

def fetch_snapshot(network_id: str, tag: str) -> pd.DataFrame:
    resp = requests.get(CITYBIKES_URL.format(network_id), timeout=30)
    resp.raise_for_status()
    stations = resp.json()['network']['stations']
    # Timestamp stored as naive UTC to match existing parquet schema
    # (pipeline does tz_localize('UTC') on ingest)
    now = pd.Timestamp.utcnow().replace(tzinfo=None)
    return pd.DataFrame([
        {
            'tag'      : tag,
            'id'       : s['id'],
            'nuid'     : s['id'],
            'name'     : s['name'],
            'latitude' : s['latitude'],
            'longitude': s['longitude'],
            'bikes'    : np.int32(s['free_bikes'] or 0),
            'free'     : float(s['empty_slots']) if s['empty_slots'] is not None else np.nan,
            'extra'    : s.get('extra', {}),
            'timestamp': now,
        }
        for s in stations
    ])

frames = []
for network_id, tag in SYSTEMS.items():
    df = fetch_snapshot(network_id, tag)
    print(f'{tag}: {len(df)} stations')
    frames.append(df)

snapshot = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(snapshot)} stations')
snapshot.head()

nextbike-berlin: 1075 stations
callabike-berlin: 631 stations

Total: 1706 stations


,tag,id,nuid,name,latitude,longitude,bikes,free,extra,timestamp
0,nextbike-berlin,00455239920d92cdc60461f8b8619fac,00455239920d92cdc60461f8b8619fac,Linienstraße/Weydingerstraße,52.53,13.41,3,3.00,"{'uid': '138056052', 'number': '10740', 'slots...",2026-06-11 09:12:25.926583
1,nextbike-berlin,008c26a668de108ce1393e5db4b4ae90,008c26a668de108ce1393e5db4b4ae90,Jelbi U Walther-Scheiber-Platz/Lefèvrestraße (...,52.47,13.33,1,6.00,"{'uid': '523749684', 'number': '13931', 'slots...",2026-06-11 09:12:25.926583
2,nextbike-berlin,008f2e8e07273aa9758cfccfae7a3fc2,008f2e8e07273aa9758cfccfae7a3fc2,Jelbi Paulstraße/Magnus-Hirschfeld-Ufer (MOA/PA),52.52,13.36,2,6.00,"{'uid': '417025384', 'number': '13768', 'slots...",2026-06-11 09:12:25.926583
3,nextbike-berlin,00cd1549463001c940bfd759958429c2,00cd1549463001c940bfd759958429c2,Jelbi Selchower Straße/Schillerpromenade (NEU/SE),52.48,13.42,1,8.00,"{'uid': '136304949', 'number': '10557', 'slots...",2026-06-11 09:12:25.926583
4,nextbike-berlin,010cd6f6ccd9d9ad025556f885b8061b,010cd6f6ccd9d9ad025556f885b8061b,S Ostkreuz (Südost),52.50,13.47,10,0.00,"{'uid': '245327817', 'number': '12309', 'slots...",2026-06-11 09:12:25.926583


In [11]:
# Validate schema matches existing parquet files
existing_file = sorted(SNAPSHOT_DIR.glob('*.parquet'))[0]
existing = pd.read_parquet(existing_file)

print('Existing schema:', existing.dtypes.to_dict())
print('\nNew snapshot schema:', snapshot.dtypes.to_dict())
print('\nColumn match:', set(existing.columns) == set(snapshot.columns))
print('\nBikes range (new):', snapshot['bikes'].min(), '–', snapshot['bikes'].max())
print('Bikes range (hist):', existing['bikes'].min(), '–', existing['bikes'].max())

Existing schema: {'tag': dtype('O'), 'id': dtype('O'), 'nuid': dtype('O'), 'name': dtype('O'), 'latitude': dtype('float64'), 'longitude': dtype('float64'), 'bikes': dtype('int32'), 'free': dtype('float64'), 'extra': dtype('O'), 'timestamp': dtype('<M8[us]')}

New snapshot schema: {'tag': dtype('O'), 'id': dtype('O'), 'nuid': dtype('O'), 'name': dtype('O'), 'latitude': dtype('float64'), 'longitude': dtype('float64'), 'bikes': dtype('int32'), 'free': dtype('float64'), 'extra': dtype('O'), 'timestamp': dtype('<M8[ns]')}

Column match: True

Bikes range (new): 0 – 25
Bikes range (hist): 0 – 35


In [12]:
# Save snapshot
out_path = SNAPSHOT_DIR / f'live_{today.isoformat()}.parquet'
snapshot.to_parquet(out_path, index=False)
print(f'Saved → {out_path.name}  ({len(snapshot)} rows)')

Saved → live_2026-06-11.parquet  (1706 rows)


## 2. Weather Forecast: Open-Meteo

In [15]:
# Fetch hourly forecast for today + tomorrow
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
client = openmeteo_requests.Client(session=retry_session)

params = {
    'latitude'     : 52.52,
    'longitude'    : 13.41,
    'hourly'       : HOURLY_VARS,
    'forecast_days': 2,
}
response = client.weather_api(FORECAST_URL, params=params)[0]
hourly   = response.Hourly()

data = {
    'timestamp': pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive='left',
    )
}
for i, var in enumerate(HOURLY_VARS):
    data[var] = hourly.Variables(i).ValuesAsNumpy()

hourly_df = pd.DataFrame(data)
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'], utc=True).dt.tz_convert(TIMEZONE)
hourly_df['date'] = hourly_df['timestamp'].dt.date

print(f'Hourly forecast: {len(hourly_df)} rows  ({hourly_df["date"].min()} → {hourly_df["date"].max()})')
hourly_df.head()

Hourly forecast: 48 rows  (2026-06-11 → 2026-06-13)


,timestamp,temperature_2m,precipitation,apparent_temperature,rain,snowfall,weather_code,wind_speed_10m,relative_humidity_2m,cloud_cover,date
0,2026-06-11 02:00:00+02:00,13.80,0.00,13.68,0.00,0.00,3.00,3.56,85.00,100.00,2026-06-11
1,2026-06-11 03:00:00+02:00,13.85,0.00,13.90,0.00,0.00,3.00,2.55,85.00,100.00,2026-06-11
2,2026-06-11 04:00:00+02:00,13.00,0.00,12.59,0.00,0.00,3.00,5.01,88.00,100.00,2026-06-11
3,2026-06-11 05:00:00+02:00,12.95,0.00,11.87,0.00,0.00,3.00,10.24,90.00,100.00,2026-06-11
4,2026-06-11 06:00:00+02:00,12.00,0.00,10.15,0.00,0.00,3.00,11.92,85.00,100.00,2026-06-11


In [17]:
# Aggregate to daily, same logic as pipeline.py
def aggregate_to_daily(hourly_df: pd.DataFrame, target_date: date) -> pd.DataFrame:
    day = hourly_df[hourly_df['date'] == target_date]
    daily = day.agg({
        'temperature_2m'      : 'mean',
        'apparent_temperature': 'mean',
        'precipitation'       : 'sum',
        'rain'                : 'sum',
        'snowfall'            : 'sum',
        'wind_speed_10m'      : 'mean',
        'cloud_cover'         : 'mean',
        'relative_humidity_2m': 'mean',
        'weather_code'        : lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
    }).to_frame().T
    daily.insert(0, 'date', pd.Timestamp(target_date))
    return daily.reset_index(drop=True)

today_weather    = aggregate_to_daily(hourly_df, today)
tomorrow_weather = aggregate_to_daily(hourly_df, tomorrow)

print('Today forecast:')
print(today_weather.to_string(index=False))
print('\nTomorrow forecast:')
print(tomorrow_weather.to_string(index=False))

Today forecast:
      date  temperature_2m  apparent_temperature  precipitation  rain  snowfall  wind_speed_10m  cloud_cover  relative_humidity_2m  weather_code
2026-06-11           15.80                 13.81           0.00  0.00      0.00           10.32        80.77                 61.05          3.00

Tomorrow forecast:
      date  temperature_2m  apparent_temperature  precipitation  rain  snowfall  wind_speed_10m  cloud_cover  relative_humidity_2m  weather_code
2026-06-12           15.23                 13.81           0.20  0.20      0.00            9.37        95.96                 69.67          3.00


In [18]:
# Validate schema matches existing weather_daily.parquet
existing_weather = pd.read_parquet(WEATHER_PATH)

print('Existing weather schema:')
print(existing_weather.dtypes)
print('\nForecast schema:')
print(tomorrow_weather.dtypes)
print('\nColumn match:', set(existing_weather.columns) == set(tomorrow_weather.columns))

Existing weather schema:
date                    datetime64[ms]
temperature_2m                 float32
apparent_temperature           float32
precipitation                  float32
rain                           float32
snowfall                       float32
wind_speed_10m                 float32
cloud_cover                    float32
relative_humidity_2m           float32
weather_code                   float32
dtype: object

Forecast schema:
date                    datetime64[s]
temperature_2m                float32
apparent_temperature          float32
precipitation                 float32
rain                          float32
snowfall                      float32
wind_speed_10m                float32
cloud_cover                   float32
relative_humidity_2m          float32
weather_code                  float32
dtype: object

Column match: True


In [19]:
# Compare tomorrow's forecast to recent historical values for sanity check
recent = existing_weather.tail(7)[['date', 'temperature_2m', 'apparent_temperature', 'precipitation']]
forecast_row = tomorrow_weather[['date', 'temperature_2m', 'apparent_temperature', 'precipitation']]

comparison = pd.concat([recent, forecast_row], ignore_index=True)
comparison['date'] = pd.to_datetime(comparison['date']).dt.date
print('Last 7 days + tomorrow forecast:')
print(comparison.to_string(index=False))

Last 7 days + tomorrow forecast:
      date  temperature_2m  apparent_temperature  precipitation
2026-04-25            9.66                  5.86           0.00
2026-04-26            9.04                  5.52           0.00
2026-04-27           10.37                  7.11           0.00
2026-04-28            9.97                  5.77           0.00
2026-04-29            9.43                  5.41           0.00
2026-04-30           12.16                  9.01           0.00
2026-05-01           11.88                  9.57           0.00
2026-06-12           15.23                 13.81           0.20


In [20]:
# Upsert tomorrow's forecast into weather_daily.parquet
existing_weather['date'] = pd.to_datetime(existing_weather['date'])
tomorrow_weather['date'] = pd.to_datetime(tomorrow_weather['date'])

# Remove existing row for tomorrow if present, then append
mask    = existing_weather['date'] != tomorrow_weather['date'].iloc[0]
updated = pd.concat([existing_weather[mask], tomorrow_weather], ignore_index=True).sort_values('date')

updated.to_parquet(WEATHER_PATH, index=False)
print(f'Weather cache updated: {len(updated)} rows  (last date: {updated["date"].max().date()})')

Weather cache updated: 487 rows  (last date: 2026-06-12)
